**Tansform Constructors Data**
1. Read bronze Constructor table
2. Keep only the columns required for analytics.
3. standardise column names using snake_case.
4. Rename columns to make them more meaningful.
5. Filer out rows where Constructor_id is null
6. Remobe duplicate records.
7. Transform values of columns Constructors _name and locality to Title case
8. write the trandformed data to silver Constructors table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
bronze_table = f"{catalog_nameone}.{bronze_schema}.constructors"
silver_table = f"{catalog_nameone}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as F


In [0]:
constructors_df = spark.table(bronze_table)

In [0]:
constructor_dropped_df = constructors_df.drop("url")

In [0]:
constructor_renamed_df = (
  constructor_dropped_df
    .withColumnsRenamed({
      "constructorId": "constructor_id",
      "name": "constructor_name"
    })
)

In [0]:
constructor_distinct_df = constructor_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
constructor_final_df = (
    constructor_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality"))) 
    
)

In [0]:
constructor_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))